# 👤 Reconnaissance Faciale Interactive

## 🎯 Objectifs
- Détecter et reconnaître des visages en temps réel
- Interface pour uploader vos propres photos
- Activation de la caméra pour reconnaissance live
- Pipeline complet de détection et reconnaissance

---
*Session 05 - SupNum Nouakchott - Formation IA & Machine Learning*

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io
import base64
import os
from pathlib import Path
import face_recognition
import pickle
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("🚀 Bibliothèques chargées avec succès!")
print("📸 Reconnaissance faciale prête!")

c:\Users\mohamed.beydia\AppData\Local\r-miniconda\envs\fr311\Lib\site-packages\face_recognition_models\__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


🚀 Bibliothèques chargées avec succès!
📸 Reconnaissance faciale prête!


## 📁 1. Configuration et Stockage des Visages

### 🗂️ Structure des données :
- **faces_database/** : Dossier pour stocker les photos de référence
- **encodings.pkl** : Fichier des encodages des visages connus

In [2]:
# Création des dossiers nécessaires
faces_dir = Path("faces_database")
faces_dir.mkdir(exist_ok=True)

print(f"📁 Dossier créé: {faces_dir}")
print("💡 Placez vos photos de référence dans ce dossier")

# Variables globales
known_face_encodings = []
known_face_names = []
encodings_file = "face_encodings.pkl"

📁 Dossier créé: faces_database
💡 Placez vos photos de référence dans ce dossier


## 🔧 2. Fonctions de Détection et Reconnaissance

In [7]:
def load_and_encode_faces():
    """Charge et encode tous les visages du dossier faces_database"""
    global known_face_encodings, known_face_names
    
    known_face_encodings = []
    known_face_names = []
    
    print("🔍 Chargement des visages de référence...")
    
    # Parcourir tous les fichiers image
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
    
    for image_path in faces_dir.iterdir():
        if image_path.suffix.lower() in image_extensions:
            try:
                # Charger l'image
                image = face_recognition.load_image_file(str(image_path))
                
                # Trouver les encodages des visages
                face_encodings = face_recognition.face_encodings(image)
                
                if face_encodings:
                    # Prendre le premier visage trouvé
                    face_encoding = face_encodings[0]
                    
                    # Utiliser le nom du fichier (sans extension) comme nom
                    name = image_path.stem
                    
                    known_face_encodings.append(face_encoding)
                    known_face_names.append(name)
                    
                    print(f"✅ Visage encodé: {name}")
                else:
                    print(f"❌ Aucun visage détecté dans: {image_path.name}")
                    
            except Exception as e:
                print(f"❌ Erreur avec {image_path.name}: {e}")
    
    # Sauvegarder les encodages
    if known_face_encodings:
        with open(encodings_file, 'wb') as f:
            pickle.dump({
                'encodings': known_face_encodings,
                'names': known_face_names
            }, f)
        print(f"💾 {len(known_face_encodings)} visages sauvegardés")
    else:
        print("⚠️ Aucun visage trouvé. Ajoutez des photos dans faces_database/")

def load_encodings():
    """Charge les encodages depuis le fichier"""
    global known_face_encodings, known_face_names
    
    if os.path.exists(encodings_file):
        with open(encodings_file, 'rb') as f:
            data = pickle.load(f)
            known_face_encodings = data['encodings']
            known_face_names = data['names']
        print(f"📂 {len(known_face_encodings)} visages chargés depuis le cache")
    else:
        print("📁 Aucun cache trouvé, chargement depuis les fichiers...")
        load_and_encode_faces()

# Charger les encodages au démarrage
load_encodings()

📂 1 visages chargés depuis le cache


## 📤 3. Interface d'Upload de Photos de Référence

In [8]:
# Widget pour uploader des photos de référence
upload_reference = widgets.FileUpload(
    accept='image/*',
    multiple=True,
    description='📤 Photos référence'
)

name_input = widgets.Text(
    placeholder='Entrez le nom de la personne',
    description='👤 Nom:',
    style={'description_width': 'initial'}
)

save_button = widgets.Button(
    description='💾 Sauvegarder',
    button_style='success'
)

output_reference = widgets.Output()

def save_reference_faces(button):
    """Sauvegarde les photos de référence uploadées"""
    with output_reference:
        clear_output(wait=True)

        # Vérif upload
        if not upload_reference.value:
            print("❌ Aucune image sélectionnée")
            return

        # Vérif nom
        if not name_input.value.strip():
            print("❌ Veuillez entrer un nom")
            return

        name = name_input.value.strip()
        saved_count = 0

        # Compatibilité v7/v8
        val = upload_reference.value
        if isinstance(val, dict):   # ipywidgets v7
            files = val.items()
        else:                       # ipywidgets v8 (tuple/list)
            files = [(f["name"], f) for f in val]

        for filename, file_info in files:
            try:
                # Sauvegarder l'image
                image_path = faces_dir / f"{name}_{saved_count}.jpg"
                with open(image_path, "wb") as f:
                    f.write(file_info["content"])

                saved_count += 1
                print(f"✅ Image sauvegardée: {image_path.name}")

            except Exception as e:
                print(f"❌ Erreur sur {filename}: {e}")

        if saved_count > 0:
            print("🔄 Rechargement des encodages...")
            load_and_encode_faces()
            print("✅ Base de données mise à jour!")

        # Reset (v7: {}, v8: ())
        upload_reference.value = ()  
        name_input.value = ""

save_button.on_click(save_reference_faces)

print("📤 Interface d'Upload de Photos de Référence")
print("=" * 50)
print("📝 Instructions:")
print("1. Sélectionnez une ou plusieurs photos de la personne")
print("2. Entrez le nom de la personne")
print("3. Cliquez sur 'Sauvegarder'")

display(widgets.VBox([
    upload_reference,
    name_input,
    save_button,
    output_reference
]))


📤 Interface d'Upload de Photos de Référence
📝 Instructions:
1. Sélectionnez une ou plusieurs photos de la personne
2. Entrez le nom de la personne
3. Cliquez sur 'Sauvegarder'


## 🖼️ 4. Test de Reconnaissance sur Photo

In [ ]:
import io
from pathlib import Path
import numpy as np
import cv2
import face_recognition
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# ---------- face recognition core ----------
def recognize_faces_in_image(image_array: np.ndarray):
    """Reconnaît les visages dans une image (RGB attendu)"""
    if len(known_face_encodings) == 0:
        return image_array, []

    # Assure RGB
    arr = image_array
    if arr.ndim == 2:
        arr = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)
    elif arr.ndim == 3 and arr.shape[2] == 4:
        arr = cv2.cvtColor(arr, cv2.COLOR_RGBA2RGB)
    # (if already 3-channel, keep as is; PIL gives RGB)

    # Détection & encodage
    face_locations = face_recognition.face_locations(arr)
    face_encodings = face_recognition.face_encodings(arr, face_locations)

    results = []
    vis = arr.copy()  # we'll draw on a copy shown by matplotlib (expects RGB)

    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
        matches = face_recognition.compare_faces(known_face_encodings, face_encoding)
        name = "Inconnu"
        confidence = 0.0

        dists = face_recognition.face_distance(known_face_encodings, face_encoding)
        if len(dists) > 0:
            i = int(np.argmin(dists))
            if matches[i]:
                name = known_face_names[i]
                confidence = 1.0 - float(dists[i])

        results.append({"name": name, "confidence": confidence,
                        "location": (top, right, bottom, left)})

        color = (0, 200, 0) if name != "Inconnu" else (200, 0, 0)  # RGB
        cv2.rectangle(vis, (left, top), (right, bottom), color, 2)
        label = f"{name} ({confidence:.1%})" if name != "Inconnu" else name
        cv2.rectangle(vis, (left, bottom - 22), (right, bottom), color, cv2.FILLED)
        cv2.putText(vis, label, (left + 4, bottom - 6),
                    cv2.FONT_HERSHEY_DUPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)

    return vis, results

# ---------- test widget ----------
upload_test = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='📸 Test photo'
)
output_test = widgets.Output()

def _first_uploaded_file(uval):
    """Return (name, dict) for first file. Works for ipywidgets v7/v8."""
    if not uval:
        return None
    if isinstance(uval, dict):               # v7
        # one file only when multiple=False
        k = next(iter(uval))
        return k, uval[k]
    else:                                    # v8 -> tuple/list of dicts
        d = uval[0]
        return d.get("name", "file"), d

def test_recognition(change):
    with output_test:
        clear_output(wait=True)

        if not upload_test.value:
            return
        if len(known_face_encodings) == 0:
            print("❌ Aucun visage de référence. Ajoutez d'abord des photos !")
            return

        try:
            # lire le fichier (v7/v8 safe)
            fname, fobj = _first_uploaded_file(upload_test.value)
            image_bytes = io.BytesIO(fobj["content"])
            image = Image.open(image_bytes).convert("RGB")  # force RGB
            image_array = np.array(image)

            # run recognition
            result_image, results = recognize_faces_in_image(image_array)

            # show side-by-side
            fig, axes = plt.subplots(1, 2, figsize=(14, 6))
            axes[0].imshow(image_array); axes[0].set_title('📸 Image originale'); axes[0].axis('off')
            axes[1].imshow(result_image); axes[1].set_title('🎯 Reconnaissance faciale'); axes[1].axis('off')
            plt.tight_layout(); plt.show()

            print("🎯 RÉSULTATS")
            print("=" * 30)
            if results:
                for i, r in enumerate(results, 1):
                    if r["name"] != "Inconnu":
                        print(f"👤 Visage {i}: {r['name']} (Confiance: {r['confidence']:.1%})")
                    else:
                        print(f"❓ Visage {i}: Personne inconnue")
            else:
                print("❌ Aucun visage détecté.")

        except Exception as e:
            print(f"❌ Erreur: {e}")

        # reset widget (v8 expects tuple)
        upload_test.value = ()

upload_test.observe(test_recognition, names='value')

print("🖼️ Test de Reconnaissance sur Photo")
print("=" * 40)
print("📝 Uploadez une photo pour tester la reconnaissance")
display(upload_test, output_test)

🖼️ Test de Reconnaissance sur Photo
📝 Uploadez une photo pour tester la reconnaissance


FileUpload(value=(), accept='image/*', description='📸 Test photo')

Output()

## 📹 5. Reconnaissance en Temps Réel avec Webcam

In [ ]:
class WebcamRecognition:
    def __init__(self):
        self.cap = None
        self.is_running = False
        
    def start_camera(self):
        """Démarre la caméra"""
        try:
            self.cap = cv2.VideoCapture(0)
            if not self.cap.isOpened():
                print("❌ Impossible d'accéder à la caméra")
                return False
            
            # Configuration de la caméra
            self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
            self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
            self.cap.set(cv2.CAP_PROP_FPS, 30)
            
            print("✅ Caméra initialisée")
            return True
        except Exception as e:
            print(f"❌ Erreur caméra: {e}")
            return False
    
    def stop_camera(self):
        """Arrête la caméra"""
        if self.cap:
            self.cap.release()
            self.cap = None
        self.is_running = False
        cv2.destroyAllWindows()
        print("📹 Caméra arrêtée")
    
    def run_recognition(self):
        """Lance la reconnaissance en temps réel"""
        if len(known_face_encodings) == 0:
            print("❌ Aucun visage de référence!")
            print("💡 Ajoutez d'abord des photos de référence")
            return
        
        if not self.start_camera():
            return
        
        self.is_running = True
        print("🎥 Reconnaissance en cours... Appuyez sur 'q' pour quitter")
        
        # Optimisation: traiter 1 frame sur 4 pour la reconnaissance
        frame_count = 0
        
        try:
            while self.is_running:
                ret, frame = self.cap.read()
                if not ret:
                    break
                
                frame_count += 1
                
                # Reconnaissance tous les 4 frames pour optimiser
                if frame_count % 4 == 0:
                    # Redimensionner pour accélérer
                    small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
                    rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
                    
                    # Détecter les visages
                    face_locations = face_recognition.face_locations(rgb_small_frame)
                    face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)
                    
                    # Reconnaissance
                    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
                        matches = face_recognition.compare_faces(known_face_encodings, face_encoding)
                        name = "Inconnu"
                        confidence = 0
                        
                        face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
                        
                        if len(face_distances) > 0:
                            best_match_index = np.argmin(face_distances)
                            if matches[best_match_index]:
                                name = known_face_names[best_match_index]
                                confidence = 1 - face_distances[best_match_index]
                        
                        # Redimensionner les coordonnées
                        top *= 4
                        right *= 4
                        bottom *= 4
                        left *= 4
                        
                        # Dessiner sur l'image complète
                        color = (0, 255, 0) if name != "Inconnu" else (0, 0, 255)
                        cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
                        
                        label = f"{name} ({confidence:.1%})" if name != "Inconnu" else name
                        cv2.rectangle(frame, (left, bottom - 35), (right, bottom), color, cv2.FILLED)
                        cv2.putText(frame, label, (left + 6, bottom - 6), 
                                   cv2.FONT_HERSHEY_DUPLEX, 0.8, (255, 255, 255), 1)
                
                # Afficher le frame
                cv2.imshow('🎥 Reconnaissance Faciale Live', frame)
                
                # Quitter avec 'q'
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
                    
        except KeyboardInterrupt:
            print("\n⏹️ Arrêt demandé")
        except Exception as e:
            print(f"❌ Erreur: {e}")
        finally:
            self.stop_camera()

# Instance de reconnaissance webcam
webcam_recognition = WebcamRecognition()

# Boutons de contrôle
start_button = widgets.Button(
    description='🎥 Démarrer Webcam',
    button_style='success'
)

stop_button = widgets.Button(
    description='⏹️ Arrêter',
    button_style='danger'
)

webcam_output = widgets.Output()

def start_webcam(button):
    """Démarre la reconnaissance webcam"""
    with webcam_output:
        clear_output(wait=True)
        print("🎥 Démarrage de la reconnaissance en temps réel...")
        print("💡 Une fenêtre va s'ouvrir - appuyez sur 'q' pour quitter")
        
    # Lancer dans un thread séparé pour éviter de bloquer Jupyter
    import threading
    thread = threading.Thread(target=webcam_recognition.run_recognition)
    thread.daemon = True
    thread.start()

def stop_webcam(button):
    """Arrête la reconnaissance webcam"""
    with webcam_output:
        webcam_recognition.is_running = False
        webcam_recognition.stop_camera()
        print("⏹️ Reconnaissance arrêtée")

start_button.on_click(start_webcam)
stop_button.on_click(stop_webcam)

print("📹 Reconnaissance Faciale en Temps Réel")
print("=" * 45)
print("⚠️ Assurez-vous d'avoir une webcam connectée")
print("💡 La fenêtre de reconnaissance s'ouvrira dans une nouvelle fenêtre")

display(widgets.HBox([start_button, stop_button]), webcam_output)

📹 Reconnaissance Faciale en Temps Réel
⚠️ Assurez-vous d'avoir une webcam connectée
💡 La fenêtre de reconnaissance s'ouvrira dans une nouvelle fenêtre


Output()

✅ Caméra initialisée
🎥 Reconnaissance en cours... Appuyez sur 'q' pour quitter
📹 Caméra arrêtée


## 📊 6. Statistiques et Gestion de la Base de Données

In [9]:
import time, numpy as np, cv2, face_recognition, ipywidgets as widgets
from IPython.display import display, clear_output

def _open_camera():
    """Essaie plusieurs backends Windows et retourne (cap, backend_name)"""
    # Essai 1 : DirectShow (le plus fiable)
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    if cap.isOpened():
        return cap, "CAP_DSHOW"
    # Essai 2 : MSMF
    cap = cv2.VideoCapture(0, cv2.CAP_MSMF)
    if cap.isOpened():
        return cap, "CAP_MSMF"
    # Essai 3 : Auto
    cap = cv2.VideoCapture(0)
    if cap.isOpened():
        return cap, "AUTO"
    return None, None

def _is_headless_opencv():
    try:
        # Les builds headless n'ont pas HighGUI → imshow plante
        # On teste via getBuildInformation ou une fenêtre test
        info = cv2.getBuildInformation()
        return "GUI:" not in info or "headless" in info.lower()
    except Exception:
        return True

class WebcamRecognition:
    def __init__(self):
        self.cap = None
        self.is_running = False
        self.backend = None

    def start_camera(self):
        self.cap, self.backend = _open_camera()
        if not self.cap:
            print("❌ Impossible d'accéder à la caméra (tous les backends ont échoué)")
            return False

        # Config standard
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        self.cap.set(cv2.CAP_PROP_FPS, 30)

        # Petit warm-up
        ok, frame = self.cap.read()
        if not ok or frame is None or frame.size == 0:
            print("⚠️ Caméra ouverte mais aucun frame lu (backend:", self.backend, ")")
            return False

        print(f"✅ Caméra initialisée (backend: {self.backend})")
        if _is_headless_opencv():
            print("⚠️ OpenCV headless détecté : 'cv2.imshow' peut ne pas fonctionner.")
            print("   Utilise la touche 'q' dans la fenêtre si elle s’ouvre; sinon, on arrêtera via le bouton.")
        return True

    def stop_camera(self):
        if self.cap:
            self.cap.release()
            self.cap = None
        self.is_running = False
        try:
            cv2.destroyAllWindows()
        except Exception:
            pass
        print("📹 Caméra arrêtée")

    def run_recognition(self):
        if len(known_face_encodings) == 0:
            print("❌ Aucun visage de référence ! Ajoutez d'abord des photos.")
            return

        if not self.start_camera():
            return

        self.is_running = True
        print("🎥 Reconnaissance en cours... Appuyez sur 'q' pour quitter")

        frame_count = 0
        try:
            while self.is_running:
                ok, frame = self.cap.read()
                if not ok or frame is None or frame.size == 0:
                    print("⚠️ Frame vide/illisible. On continue…")
                    time.sleep(0.05)
                    continue

                # Effet miroir pour se voir naturellement
                frame = cv2.flip(frame, 1)

                frame_count += 1
                draw_frame = frame.copy()

                if frame_count % 4 == 0:
                    # Downscale pour accélérer
                    small = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
                    rgb_small = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)

                    face_locs = face_recognition.face_locations(rgb_small)
                    face_encs = face_recognition.face_encodings(rgb_small, face_locs)

                    for (top, right, bottom, left), face_enc in zip(face_locs, face_encs):
                        matches = face_recognition.compare_faces(known_face_encodings, face_enc)
                        name, conf = "Inconnu", 0.0

                        dists = face_recognition.face_distance(known_face_encodings, face_enc)
                        if len(dists) > 0:
                            i = int(np.argmin(dists))
                            if matches[i]:
                                name = known_face_names[i]
                                conf = 1.0 - float(dists[i])

                        # Remise à l’échelle
                        top, right, bottom, left = top*4, right*4, bottom*4, left*4

                        color = (0, 255, 0) if name != "Inconnu" else (0, 0, 255)
                        cv2.rectangle(draw_frame, (left, top), (right, bottom), color, 2)
                        label = f"{name} ({conf:.0%})" if name != "Inconnu" else name
                        cv2.rectangle(draw_frame, (left, bottom - 24), (right, bottom), color, cv2.FILLED)
                        cv2.putText(draw_frame, label, (left + 4, bottom - 6),
                                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)

                # Affichage (si HighGUI dispo)
                try:
                    cv2.imshow('🎥 Reconnaissance Faciale Live', draw_frame)
                    if cv2.waitKey(1) & 0xFF == ord('q'):
                        break
                except Exception as e:
                    # Build headless: on ne peut pas imshow → on log et on arrête proprement
                    print("⚠️ OpenCV ne peut pas afficher de fenêtre (headless).", e)
                    print("   La détection tourne, mais sans affichage. Arrêt…")
                    break

        except KeyboardInterrupt:
            print("\n⏹️ Arrêt demandé")
        except Exception as e:
            print(f"❌ Erreur: {e}")
        finally:
            self.stop_camera()

# --- Widgets de contrôle
webcam_recognition = WebcamRecognition()

start_button = widgets.Button(description='🎥 Démarrer Webcam', button_style='success')
stop_button  = widgets.Button(description='⏹️ Arrêter',          button_style='danger')
webcam_output = widgets.Output()

def start_webcam(_):
    with webcam_output:
        clear_output(wait=True)
        print("🎥 Démarrage… Une fenêtre va s'ouvrir (appuyez sur 'q' pour quitter).")
    import threading
    t = threading.Thread(target=webcam_recognition.run_recognition, daemon=True)
    t.start()

def stop_webcam(_):
    with webcam_output:
        webcam_recognition.is_running = False
        webcam_recognition.stop_camera()
        print("⏹️ Reconnaissance arrêtée")

start_button.on_click(start_webcam)
stop_button.on_click(stop_webcam)

display(widgets.HBox([start_button, stop_button]), webcam_output)


Output()

## 🎓 7. Guide d'Utilisation et Conseils

### 📝 Étapes pour utiliser ce notebook :

1. **📤 Ajouter des photos de référence** :
   - Uploadez des photos claires de chaque personne
   - Utilisez des photos avec un seul visage visible
   - Variez les angles et expressions

2. **🖼️ Tester sur des photos** :
   - Uploadez une photo pour voir la reconnaissance
   - Vérifiez la précision et la confiance

3. **📹 Reconnaissance en temps réel** :
   - Cliquez sur "Démarrer Webcam"
   - Positionnez-vous face à la caméra
   - Appuyez sur 'q' pour quitter

### 💡 Conseils pour de meilleurs résultats :

- **Photos de qualité** : Utilisez des images nettes et bien éclairées
- **Visages frontaux** : Les photos de face donnent de meilleurs résultats
- **Plusieurs angles** : Ajoutez plusieurs photos par personne
- **Bon éclairage** : Évitez les contre-jours et ombres fortes
- **Distance appropriée** : Le visage doit occuper une bonne partie de l'image

### ⚠️ Limitations :

- Fonctionne mieux avec des visages frontaux
- Sensible aux changements d'éclairage
- Peut confondre des personnes similaires
- Performance dépend de la qualité des photos de référence

---
**🎉 Amusez-vous avec la reconnaissance faciale !**